In [ ]:
import os
_cuda_bin = r'C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0\bin'
if os.path.isdir(_cuda_bin):
    os.add_dll_directory(_cuda_bin)
    if os.path.isdir(_cuda_bin + r'\x64'):
        os.add_dll_directory(_cuda_bin + r'\x64')

import numpy as np
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def set_seed(seed=2025):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

In [ ]:
MANIFEST_PATH = 'ScAlN_dataset_3class/split_manifest.json'
DATA_DIR = 'ScAlN_dataset_3class'
LABEL_NAMES = ['Streaky', 'Transition', 'Spotty']

def load_data_from_manifest():
    print(f"Loading splits from: {MANIFEST_PATH}")
    X_data = np.load(os.path.join(DATA_DIR, 'X_ScAlN_3class.npy'))
    y_data = np.load(os.path.join(DATA_DIR, 'y_ScAlN_3class_labeled.npy')).flatten()
    print(f"Raw data: X={X_data.shape}, y={y_data.shape}")

    if X_data.ndim == 3:
        X_data = np.expand_dims(X_data, axis=1)
    if X_data.shape[1] == 1:
        X_data = np.repeat(X_data, 3, axis=1)
    X_data = X_data.astype(np.float32)
    if X_data.max() > 1.0:
        X_data = X_data / 255.0
    if y_data.min() == 1:
        y_data = y_data - 1

    with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
        manifest = json.load(f)

    splits = {}
    for split_name in ['train', 'val', 'test']:
        idx = manifest['splits'][split_name]['frame_indices']
        X = torch.FloatTensor(X_data[idx])
        y = torch.LongTensor(y_data[idx])
        splits[split_name] = (X, y)
        unique, counts = np.unique(y_data[idx], return_counts=True)
        dist = {LABEL_NAMES[k]: int(v) for k, v in zip(unique, counts)}
        print(f"  {split_name}: {len(idx)} frames, dist={dist}")

    train_y = y_data[manifest['splits']['train']['frame_indices']]
    class_weights = compute_class_weight('balanced', classes=np.unique(train_y), y=train_y)
    return splits, torch.FloatTensor(class_weights)


class RHEEDDataset(Dataset):
    def __init__(self, X, y, training=True):
        self.X, self.y = X, y
        self.training = training
    def __getitem__(self, idx):
        img = self.X[idx].clone()
        if self.training:
            if torch.rand(1) < 0.3:
                img = img * (0.8 + 0.4 * torch.rand(1).item())
            if torch.rand(1) < 0.3:
                img = img + torch.randn_like(img) * 0.02
            if torch.rand(1) < 0.2:
                _, h, w = img.shape
                eh = int(h * (0.05 + 0.1 * torch.rand(1).item()))
                ew = int(w * (0.05 + 0.1 * torch.rand(1).item()))
                y0 = torch.randint(0, h - eh, (1,)).item()
                x0 = torch.randint(0, w - ew, (1,)).item()
                img[:, y0:y0+eh, x0:x0+ew] = torch.rand(3, eh, ew)
            img = torch.clamp(img, 0, 1)
        return img, self.y[idx]
    def __len__(self):
        return len(self.X)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=(32, 64), patch_size=(8, 8), in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size[0] // patch_size[0]) * (img_size[1] // patch_size[1])
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class TransformerBlockVit(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim), nn.Dropout(dropout)
        )
    def forward(self, x):
        x1 = self.norm1(x)
        attn_out, _ = self.attn(x1, x1, x1)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class TransformerOnly(nn.Module):
    def __init__(self, img_size=(32, 64), patch_size=(8, 8), in_channels=3,
                 num_classes=3, embed_dim=256, depth=8, num_heads=8,
                 mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlockVit(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.classifier(x[:, 0])

In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes=3, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.weight = weight
    def forward(self, pred, target):
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        log_prob = F.log_softmax(pred, dim=1)
        loss = -true_dist * log_prob
        if self.weight is not None:
            w = self.weight.to(pred.device)[target]
            loss = loss.sum(1) * w
        else:
            loss = loss.sum(1)
        return loss.mean()

In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    class_correct = [0, 0, 0]
    class_total = [0, 0, 0]
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()
            for i in range(3):
                class_total[i] += (targets == i).sum().item()
                class_correct[i] += ((pred == i) & (targets == i)).sum().item()
    recalls = [class_correct[i] / max(class_total[i], 1) for i in range(3)]
    return total_loss / len(loader), correct / total, recalls

In [ ]:
def main():
    set_seed(2025)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    splits, class_weights = load_data_from_manifest()
    class_weights = class_weights.to(device)

    X_train, y_train = splits['train']
    X_val, y_val = splits['val']
    X_test, y_test = splits['test']

    train_dataset = RHEEDDataset(X_train, y_train, training=True)
    val_dataset = RHEEDDataset(X_val, y_val, training=False)
    test_dataset = RHEEDDataset(X_test, y_test, training=False)

    BATCH_SIZE = 32
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, pin_memory=True)

    print(f"Dataset: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

    model = TransformerOnly(
        img_size=(32, 64), patch_size=(8, 8), in_channels=3,
        num_classes=3, embed_dim=256, depth=8, num_heads=8,
        mlp_ratio=4, dropout=0.1
    ).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Params: {total_params:,} ({total_params/1e6:.2f}M)")

    criterion = LabelSmoothingCE(num_classes=3, smoothing=0.1, weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-7)
    # 5-epoch warmup then cosine decay
    warmup = LinearLR(optimizer, start_factor=0.01, total_iters=5)
    cosine = CosineAnnealingLR(optimizer, T_max=125, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[5])

    EPOCHS = 130
    PATIENCE = 35
    best_val_acc = 0
    counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_model_path = 'results/transformer_v2/best_model.pth'
    os.makedirs('results/transformer_v2', exist_ok=True)

    print(f"\nStarting Transformer-Only (ViT) training...")
    train_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            _, pred = outputs.max(1)
            total += targets.size(0)
            correct += pred.eq(targets).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        val_loss, val_acc, recalls = validate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(float(val_loss))
        history['train_acc'].append(float(train_acc))
        history['val_acc'].append(float(val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            counter = 0
        else:
            counter += 1

        if (epoch + 1) % 10 == 0 or epoch < 5:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch [{epoch+1}/{EPOCHS}] Train: {train_acc:.4f} | Val: {val_acc:.4f} | '
                  f'Recalls: {recalls[0]:.3f}/{recalls[1]:.3f}/{recalls[2]:.3f} | LR: {lr:.2e}')

        if counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    training_time = time.time() - train_start
    print(f"Training time: {training_time:.1f}s ({training_time/60:.1f}min)")

    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    test_loss, test_acc, test_recalls = validate(model, test_loader, criterion, device)

    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs.to(device))
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())

    print(f"\n{'='*50}")
    print(f"Transformer-Only test results")
    print(f"{'='*50}")
    print(f"Best val accuracy: {best_val_acc:.4f}")
    print(f"Test accuracy: {test_acc:.4f}")
    print(classification_report(all_targets, all_preds, target_names=LABEL_NAMES))

    with open('results/transformer_v2/training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    cm = confusion_matrix(all_targets, all_preds)
    report = classification_report(all_targets, all_preds, target_names=LABEL_NAMES, output_dict=True)
    test_report = {
        'model_name': 'transformer',
        'test_accuracy': float(test_acc),
        'best_val_accuracy': float(best_val_acc),
        'test_loss': float(test_loss),
        'per_class_recall': {LABEL_NAMES[i]: float(test_recalls[i]) for i in range(3)},
        'confusion_matrix': cm.tolist(),
        'classification_report': report,
        'total_params': total_params,
        'training_time_seconds': float(training_time),
        'training_time_minutes': float(training_time / 60),
        'epochs_trained': len(history['train_loss']),
        'test_set_size': len(test_dataset),
    }
    with open('results/transformer_v2/test_report.json', 'w') as f:
        json.dump(test_report, f, indent=2, ensure_ascii=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs_range = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs_range, history['train_loss'], label='Train')
    axes[0].plot(epochs_range, history['val_loss'], label='Val')
    axes[0].set_title('Transformer Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs_range, history['train_acc'], label='Train')
    axes[1].plot(epochs_range, history['val_acc'], label='Val')
    axes[1].set_title('Transformer Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    axes[2].set_title(f'Confusion Matrix (Acc={test_acc:.4f})')
    plt.tight_layout()
    plt.savefig('results/transformer_v2/training_curves.png', dpi=300)
    plt.show()

    return history, test_report

In [ ]:
if __name__ == "__main__":
    history, test_report = main()